# NLP 和大语言模型实践

本notebook包含以下内容：
1. Python基础：函数参数 (*args, **kwargs)
2. 中文自然语言处理 (NLP)
3. BERT模型微调
4. LLaMA 3.2模型微调
5. Gradio交互界面

## 1. Python基础：函数参数

理解 `*args` 和 `**kwargs` 的使用

### *args - 可变位置参数

`*args` 允许函数接受任意数量的位置参数，这些参数会被收集到一个元组中。

In [ ]:
def example(*args, **kwargs):
    """
    演示 *args 和 **kwargs 的使用
    
    *args: 收集所有位置参数到一个元组
    **kwargs: 收集所有关键字参数到一个字典
    """
    print("位置参数 (args):")
    print("  值:", args)
    print("  类型:", type(args))
    
    print("\n关键字参数 (kwargs):")
    print("  值:", kwargs)
    print("  类型:", type(kwargs))

In [ ]:
# 示例1: 传入列表作为单个位置参数
print("=" * 50)
print("示例1: 传入列表")
print("=" * 50)
example([1, 2, 3, 4, 5])

In [ ]:
# 示例2: 传入多个位置参数和关键字参数
print("\n" + "=" * 50)
print("示例2: 混合参数")
print("=" * 50)
example(1, 2, name=3)

### 实用示例

In [ ]:
def sum_all(*numbers):
    """计算所有传入数字的和"""
    return sum(numbers)

print("sum_all(1, 2, 3):", sum_all(1, 2, 3))
print("sum_all(1, 2, 3, 4, 5):", sum_all(1, 2, 3, 4, 5))
print("sum_all(*range(1, 11)):", sum_all(*range(1, 11)))  # 1到10的和

In [ ]:
def greet(greeting, *names, **options):
    """问候多个人"""
    separator = options.get('separator', ', ')
    uppercase = options.get('uppercase', False)
    
    names_str = separator.join(names)
    if uppercase:
        names_str = names_str.upper()
    
    return f"{greeting} {names_str}!"

print(greet("Hello", "Alice", "Bob", "Charlie"))
print(greet("Hi", "Alice", "Bob", separator=" and "))
print(greet("Welcome", "Alice", "Bob", uppercase=True))

## 2. 中文自然语言处理 (NLP)

使用jieba和sklearn实现中文文本处理

### 安装依赖

In [ ]:
# 如果没有安装，请取消注释运行
# !pip install jieba scikit-learn networkx

In [ ]:
import re
from collections import Counter
import jieba
import jieba.analyse
import jieba.posseg as pseg
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

### ChineseNLPProcessor 类

一个综合的中文NLP处理类，包含以下功能：
- 文本清理
- 分词
- 关键词提取
- 文本相似度计算
- 简单情感分析
- 文本摘要生成
- 词性标注

In [ ]:
class ChineseNLPProcessor:
    def __init__(self):
        # 停用词列表
        self.stop_words = set([
            '的', '了', '在', '是', '我', '有', '和', '就', '不', '人', 
            '都', '一', '一个', '上', '也', '很', '到', '说', '要', '去', 
            '你', '会', '着', '没有', '看', '好', '自己', '这'
        ])
        
    def text_cleanup(self, text):
        """
        文本清理：去除特殊字符、多余空格等
        """
        # 移除HTML标签
        text = re.sub(r'<[^>]+>', '', text)
        # 移除多余空格
        text = re.sub(r'\s+', ' ', text)
        # 移除URL
        text = re.sub(r'http\S+|www.\S+', '', text)
        # 移除表情符号
        text = re.sub(r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]', '', text)
        return text.strip()
    
    def segment_text(self, text):
        """
        分词处理
        """
        words = jieba.cut(text)
        return [word for word in words if word not in self.stop_words and len(word.strip()) > 0]
    
    def extract_keywords(self, text, top_n=5):
        """
        提取关键词：使用TF-IDF算法
        """
        keywords = jieba.analyse.extract_tags(text, topK=top_n, withWeight=True)
        return keywords
    
    def calculate_similarity(self, text1, text2):
        """
        计算两段文本的相似度
        """
        vectorizer = TfidfVectorizer()
        try:
            tfidf_matrix = vectorizer.fit_transform([text1, text2])
            similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
            return round(similarity, 4)
        except:
            return 0.0
    
    def sentiment_analysis_simple(self, text):
        """
        简单情感分析（基于情感词典）
        """
        # 示例情感词典
        positive_words = set(['喜欢', '好', '优秀', '棒', '完美', '快乐', '开心', '出色'])
        negative_words = set(['差', '糟糕', '讨厌', '失望', '难过', '不好', '低劣', '痛苦'])
        
        words = self.segment_text(text)
        positive_count = sum(1 for word in words if word in positive_words)
        negative_count = sum(1 for word in words if word in negative_words)
        
        if positive_count > negative_count:
            return 'positive'
        elif negative_count > positive_count:
            return 'negative'
        else:
            return 'neutral'
    
    def extract_summary(self, text, num_sentences=3):
        """
        提取文本摘要（基于TextRank算法）
        """
        # 分句
        sentences = re.split(r'[。！？]', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        if not sentences:
            return ""
            
        # 构建图
        graph = nx.Graph()
        
        # 添加节点和边
        for i, sent_i in enumerate(sentences):
            for j, sent_j in enumerate(sentences):
                if i < j:
                    similarity = self.calculate_similarity(sent_i, sent_j)
                    if similarity > 0:
                        graph.add_edge(i, j, weight=similarity)
        
        # 使用PageRank算法
        scores = nx.pagerank(graph, alpha=0.85)
        
        # 选择得分最高的句子
        ranked_sentences = [(sent, scores.get(i, 0)) 
                          for i, sent in enumerate(sentences)]
        ranked_sentences.sort(key=lambda x: x[1], reverse=True)
        
        # 按原文顺序重排
        selected_sentences = ranked_sentences[:num_sentences]
        selected_sentences.sort(key=lambda x: sentences.index(x[0]))
        
        summary = '。'.join(s[0] for s in selected_sentences) + '。'
        return summary

    def word_segmentation_analysis(self, text):
        """
        词性标注分析
        """
        words = pseg.cut(text)
        return [(word.word, word.flag) for word in words]

### 使用示例

In [ ]:
# 创建NLP处理器实例
nlp = ChineseNLPProcessor()

# 示例文本
text = """
今天天气真不错，阳光明媚，我和朋友去公园散步。
公园里人很多，有人在跑步，有人在打太极，还有人在下棋。
这样的周末生活真是惬意，让人感到很放松和开心。
"""

In [ ]:
# 1. 清理文本
cleaned_text = nlp.text_cleanup(text)
print("清理后的文本:")
print(cleaned_text)

In [ ]:
# 2. 分词
words = nlp.segment_text(cleaned_text)
print("\n分词结果:")
print(' / '.join(words))

In [ ]:
# 3. 提取关键词
keywords = nlp.extract_keywords(cleaned_text)
print("\n关键词 (词, 权重):")
for word, weight in keywords:
    print(f"  {word}: {weight:.4f}")

In [ ]:
# 4. 情感分析
sentiment = nlp.sentiment_analysis_simple(cleaned_text)
print("\n情感分析结果:", sentiment)

In [ ]:
# 5. 生成摘要
summary = nlp.extract_summary(cleaned_text, num_sentences=2)
print("\n文本摘要:")
print(summary)

In [ ]:
# 6. 词性标注
pos_tags = nlp.word_segmentation_analysis(cleaned_text)
print("\n词性标注 (前15个):")
for word, pos in pos_tags[:15]:
    print(f"  {word} ({pos})")

In [ ]:
# 7. 文本相似度
text1 = "今天天气很好，适合出去玩"
text2 = "今天阳光明媚，适合户外活动"
text3 = "明天要下雨，记得带伞"

print("\n文本相似度:")
print(f"文本1 vs 文本2: {nlp.calculate_similarity(text1, text2):.4f}")
print(f"文本1 vs 文本3: {nlp.calculate_similarity(text1, text3):.4f}")
print(f"文本2 vs 文本3: {nlp.calculate_similarity(text2, text3):.4f}")

## 3. BERT 模型微调

使用Transformers库对BERT进行情感分类任务的微调

### 安装依赖

In [ ]:
# 如果没有安装，请取消注释运行
# !pip install torch transformers scikit-learn

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification, AdamW
from transformers import get_linear_schedule_with_warmup
import numpy as np
from sklearn.model_selection import train_test_split
import logging
import time

### 自定义数据集类

In [ ]:
class ChineseSentimentDataset(Dataset):
    """中文情感分析数据集"""
    
    def __init__(self, texts, labels, tokenizer, max_len=128):
        """
        参数:
            texts: 文本列表
            labels: 标签列表
            tokenizer: BERT tokenizer
            max_len: 最大序列长度
        """
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        # 使用tokenizer进行编码
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=True,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'token_type_ids': encoding['token_type_ids'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

### BERT微调器类

In [ ]:
class BertFineTuner:
    """BERT微调器"""
    
    def __init__(self, model_name='bert-base-chinese', num_labels=2):
        """
        初始化BERT微调器
        
        参数:
            model_name: 预训练模型名称
            num_labels: 分类标签数量
        """
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"使用设备: {self.device}")
        
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = BertForSequenceClassification.from_pretrained(
            model_name,
            num_labels=num_labels
        )
        self.model.to(self.device)
        
        # 设置日志
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)

    def prepare_data(self, texts, labels, test_size=0.2, max_len=128, batch_size=16):
        """准备训练和验证数据"""
        # 划分训练集和验证集
        train_texts, val_texts, train_labels, val_labels = train_test_split(
            texts, labels, test_size=test_size, random_state=42
        )

        # 创建数据集
        train_dataset = ChineseSentimentDataset(
            train_texts, train_labels, self.tokenizer, max_len
        )
        val_dataset = ChineseSentimentDataset(
            val_texts, val_labels, self.tokenizer, max_len
        )

        # 创建数据加载器
        train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

        return train_dataloader, val_dataloader

    def train(self, train_dataloader, val_dataloader, epochs=3, learning_rate=2e-5):
        """训练模型"""
        # 优化器
        optimizer = AdamW(self.model.parameters(), lr=learning_rate)

        # 学习率调度器
        total_steps = len(train_dataloader) * epochs
        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=0, num_training_steps=total_steps
        )

        # 训练循环
        for epoch in range(epochs):
            self.logger.info(f'\n=== Epoch {epoch + 1}/{epochs} ===')
            start_time = time.time()

            # 训练阶段
            self.model.train()
            train_loss = 0
            
            for batch_idx, batch in enumerate(train_dataloader):
                # 将数据移到设备
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                token_type_ids = batch['token_type_ids'].to(self.device)
                labels = batch['labels'].to(self.device)

                # 清零梯度
                optimizer.zero_grad()

                # 前向传播
                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids,
                    labels=labels
                )

                loss = outputs.loss
                train_loss += loss.item()

                # 反向传播
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                
                if (batch_idx + 1) % 10 == 0:
                    self.logger.info(f'  Batch {batch_idx + 1}/{len(train_dataloader)}, Loss: {loss.item():.4f}')

            avg_train_loss = train_loss / len(train_dataloader)

            # 验证阶段
            self.model.eval()
            val_loss = 0
            val_accuracy = 0
            val_steps = 0

            with torch.no_grad():
                for batch in val_dataloader:
                    input_ids = batch['input_ids'].to(self.device)
                    attention_mask = batch['attention_mask'].to(self.device)
                    token_type_ids = batch['token_type_ids'].to(self.device)
                    labels = batch['labels'].to(self.device)

                    outputs = self.model(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        token_type_ids=token_type_ids,
                        labels=labels
                    )

                    loss = outputs.loss
                    val_loss += loss.item()

                    # 计算准确率
                    logits = outputs.logits
                    predictions = torch.argmax(logits, dim=-1)
                    val_accuracy += (predictions == labels).sum().item()
                    val_steps += len(labels)

            avg_val_loss = val_loss / len(val_dataloader)
            val_accuracy = val_accuracy / val_steps

            # 记录训练信息
            time_elapsed = time.time() - start_time
            self.logger.info(f'\nEpoch {epoch + 1} 结果:')
            self.logger.info(f'  训练损失: {avg_train_loss:.4f}')
            self.logger.info(f'  验证损失: {avg_val_loss:.4f}')
            self.logger.info(f'  验证准确率: {val_accuracy:.4f}')
            self.logger.info(f'  耗时: {time_elapsed:.2f}秒')

    def save_model(self, path):
        """保存模型"""
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        print(f"模型已保存到: {path}")

    def load_model(self, path):
        """加载模型"""
        self.model = BertForSequenceClassification.from_pretrained(path)
        self.tokenizer = BertTokenizer.from_pretrained(path)
        self.model.to(self.device)
        print(f"模型已从 {path} 加载")

    def predict(self, texts, max_len=128):
        """预测新数据"""
        self.model.eval()
        predictions = []
        probabilities = []

        with torch.no_grad():
            for text in texts:
                encoding = self.tokenizer.encode_plus(
                    text,
                    add_special_tokens=True,
                    max_length=max_len,
                    return_token_type_ids=True,
                    padding='max_length',
                    truncation=True,
                    return_attention_mask=True,
                    return_tensors='pt'
                )

                input_ids = encoding['input_ids'].to(self.device)
                attention_mask = encoding['attention_mask'].to(self.device)
                token_type_ids = encoding['token_type_ids'].to(self.device)

                outputs = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    token_type_ids=token_type_ids
                )

                logits = outputs.logits
                probs = torch.softmax(logits, dim=-1)
                prediction = torch.argmax(logits, dim=-1)
                
                predictions.append(prediction.item())
                probabilities.append(probs.cpu().numpy()[0])

        return predictions, probabilities

### 使用示例

In [ ]:
# 示例数据（实际使用时应该有更多数据）
texts = [
    "这个产品质量很好，我很满意",
    "服务态度差，不推荐购买",
    "价格合理，值得购买",
    "产品性能优秀，超出预期",
    "太差了，完全是浪费钱",
    "物流速度快，包装完好",
    "客服很热情，解决了我的问题",
    "质量有问题，退货也不给处理",
    "性价比很高，推荐购买",
    "产品有质量问题，很失望",
    "非常满意，会继续购买",
    "完全不值这个价格",
    "质量不错，用着很舒服",
    "售后服务很差劲",
    "物有所值，很喜欢"
]

# 1表示正面，0表示负面
labels = [1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1]

print(f"数据集大小: {len(texts)}")
print(f"正面样本: {sum(labels)}")
print(f"负面样本: {len(labels) - sum(labels)}")

In [ ]:
# 初始化微调器（首次运行会下载模型）
# 注意：如果没有网络连接或无法下载模型，这一步会失败
# finetuner = BertFineTuner(model_name='bert-base-chinese', num_labels=2)

In [ ]:
# 准备数据
# train_dataloader, val_dataloader = finetuner.prepare_data(
#     texts, labels, test_size=0.2, max_len=128, batch_size=4
# )

In [ ]:
# 训练模型
# finetuner.train(train_dataloader, val_dataloader, epochs=3, learning_rate=2e-5)

In [ ]:
# 保存模型
# finetuner.save_model('finetuned-bert-sentiment')

In [ ]:
# 预测新数据
# new_texts = [
#     "这个产品非常好用",
#     "质量很差，退货了",
#     "还不错，可以考虑"
# ]
# predictions, probabilities = finetuner.predict(new_texts)

# for text, pred, prob in zip(new_texts, predictions, probabilities):
#     sentiment = "正面" if pred == 1 else "负面"
#     print(f"\n文本: {text}")
#     print(f"预测: {sentiment}")
#     print(f"概率: 负面={prob[0]:.4f}, 正面={prob[1]:.4f}")

## 4. LLaMA 3.2 模型微调

使用LoRA (Low-Rank Adaptation) 技术微调LLaMA 3.2模型

### 安装依赖

In [ ]:
# 安装所需库
# !pip install transformers datasets peft bitsandbytes accelerate

### 关键概念

**LoRA (Low-Rank Adaptation)**
- 一种参数高效的微调方法
- 只训练少量额外参数，大大降低计算成本
- 适合在消费级GPU上微调大模型

**4-bit量化**
- 使用BitsAndBytes库进行4位量化
- 显著降低模型内存占用
- 在保持性能的同时提高训练效率

**PEFT (Parameter-Efficient Fine-Tuning)**
- 参数高效微调框架
- 支持LoRA、Prefix Tuning等多种方法

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftModel
)
import os

### 数据准备函数

In [ ]:
def prepare_dataset(tokenizer, dataset_name="databricks/databricks-dolly-15k", max_samples=100):
    """
    准备并处理数据集
    
    参数:
        tokenizer: 分词器
        dataset_name: 数据集名称
        max_samples: 最大样本数（用于快速测试）
    """
    # 加载数据集
    dataset = load_dataset(dataset_name, split="train")
    
    # 限制样本数量（可选）
    if max_samples:
        dataset = dataset.select(range(min(max_samples, len(dataset))))
    
    print(f"数据集大小: {len(dataset)}")
    print("数据示例:", dataset[0])

    def format_conversation(example):
        """格式化对话"""
        instruction = example.get('instruction', '')
        context = example.get('context', '')
        response = example.get('response', '')

        conversation = f"""### Instruction: {instruction}

### Input: {context if context else 'No additional context provided.'}

### Response: {response}"""
        return conversation

    def preprocess_function(examples):
        """预处理函数"""
        conversations = []
        
        # 处理批次中的每个样本
        for i in range(len(examples['instruction'])):
            example = {
                'instruction': examples['instruction'][i],
                'context': examples['context'][i] if 'context' in examples else '',
                'response': examples['response'][i]
            }
            conversations.append(format_conversation(example))

        # 分词
        tokenized = tokenizer(
            conversations,
            truncation=True,
            max_length=512,
            padding="max_length",
            return_tensors="pt"
        )

        # 设置标签
        tokenized["labels"] = tokenized["input_ids"].clone()

        return tokenized

    # 应用预处理
    tokenized_dataset = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset.column_names
    )

    return tokenized_dataset

### 模型准备函数

In [ ]:
def prepare_model(model_id, token=None):
    """
    准备模型进行训练
    
    参数:
        model_id: 模型ID
        token: HuggingFace访问令牌
    """
    # 1. 量化配置
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,              # 4位量化
        bnb_4bit_quant_type="nf4",      # NormalFloat 4位
        bnb_4bit_compute_dtype=torch.float16,  # 计算数据类型
        bnb_4bit_use_double_quant=True  # 双重量化
    )

    # 2. 加载基础模型
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        token=token,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    # 3. 加载tokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        token=token,
        trust_remote_code=True
    )
    tokenizer.pad_token = tokenizer.eos_token

    # 4. 为LoRA准备模型
    model = prepare_model_for_kbit_training(model)

    # 5. LoRA配置
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=8,                    # LoRA秩
        lora_alpha=32,          # LoRA alpha参数
        lora_dropout=0.05,      # Dropout概率
        target_modules=[        # 需要应用LoRA的模块
            "q_proj",
            "v_proj",
            "k_proj",
            "o_proj",
        ],
    )

    # 6. 获取PEFT模型
    model = get_peft_model(model, lora_config)
    
    # 打印可训练参数
    model.print_trainable_parameters()

    return model, tokenizer

### 训练函数

In [ ]:
def train_model(model, tokenizer, train_dataset, output_dir="./llama_finetuned"):
    """
    训练模型
    
    参数:
        model: 模型
        tokenizer: 分词器
        train_dataset: 训练数据集
        output_dir: 输出目录
    """
    # 1. 训练参数配置
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        weight_decay=0.001,
        logging_steps=10,
        save_steps=100,
        save_total_limit=3,
        fp16=True,
        save_safetensors=True,
        save_strategy="steps",
    )

    # 2. 数据整理器
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False  # 不使用掩码语言模型
    )

    # 3. 初始化训练器
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=data_collator,
    )

    # 4. 开始训练
    print("开始训练...")
    trainer.train()

    # 5. 保存模型
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"模型已保存到: {output_dir}")

    return trainer

### 推理函数

In [ ]:
def inference(model_id, model_path, token=None, prompt=None):
    """
    使用微调后的模型进行推理
    
    参数:
        model_id: 基础模型ID
        model_path: 微调模型路径
        token: HuggingFace访问令牌
        prompt: 输入提示
    """
    try:
        # 1. 加载基础模型
        base_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            token=token,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.float16,
        )

        # 2. 加载PEFT模型
        model = PeftModel.from_pretrained(
            base_model,
            model_path,
            torch_dtype=torch.float16,
            device_map="auto"
        )

        # 3. 加载tokenizer
        tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            token=token,
            trust_remote_code=True
        )

        # 4. 默认测试输入
        if prompt is None:
            prompt = """### Instruction: Tell me about artificial intelligence

### Input: I want to learn about AI

### Response:"""

        # 5. 生成回答
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            num_return_sequences=1,
            do_sample=True,
            top_p=0.95,
        )

        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response

    except Exception as e:
        print(f"推理错误: {e}")
        import traceback
        traceback.print_exc()
        return None

### 完整流程示例

In [ ]:
# 注意：运行此部分需要：
# 1. HuggingFace账号和访问令牌
# 2. 足够的GPU内存（建议至少16GB）
# 3. 网络连接以下载模型

# 配置参数
# MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
# HF_TOKEN = "your_huggingface_token_here"  # 替换为你的token
# OUTPUT_DIR = "./finetuned_llama"

# # 1. 准备模型和tokenizer
# print("正在准备模型...")
# model, tokenizer = prepare_model(MODEL_ID, HF_TOKEN)

# # 2. 准备数据集（使用少量样本快速测试）
# print("\n正在准备数据集...")
# train_dataset = prepare_dataset(tokenizer, max_samples=100)

# # 3. 训练模型
# print("\n开始训练...")
# trainer = train_model(model, tokenizer, train_dataset, OUTPUT_DIR)

# # 4. 测试微调后的模型
# print("\n测试微调后的模型...")
# response = inference(MODEL_ID, OUTPUT_DIR, HF_TOKEN)
# if response:
#     print("\n生成的回答:")
#     print(response)

### 关键要点总结

1. **内存优化**
   - 使用4位量化减少内存占用
   - LoRA只训练少量参数
   - 梯度累积减少批次大小

2. **训练效率**
   - 混合精度训练(fp16)
   - 合适的学习率和批次大小
   - 定期保存检查点

3. **模型保存**
   - PEFT模型只保存LoRA权重
   - 推理时需要加载基础模型+LoRA权重
   - 节省存储空间

## 5. Gradio 交互界面

使用Gradio快速创建机器学习模型的Web界面

### 安装Gradio

In [ ]:
# !pip install gradio

In [ ]:
import gradio as gr

### 示例1：简单的问候应用

In [ ]:
def greet(name, intensity):
    """
    生成问候语
    
    参数:
        name: 名字
        intensity: 强度（重复次数）
    """
    return "Hello, " + name + "!" * int(intensity)

# 创建Gradio界面
demo1 = gr.Interface(
    fn=greet,
    inputs=[
        gr.Textbox(label="名字", placeholder="请输入你的名字"),
        gr.Slider(minimum=1, maximum=10, step=1, value=1, label="强度")
    ],
    outputs=gr.Textbox(label="问候语"),
    title="简单问候应用",
    description="输入你的名字和强度，生成个性化的问候语"
)

# 启动界面（在notebook中）
# demo1.launch()

### 示例2：中文情感分析界面

In [ ]:
# 使用前面定义的ChineseNLPProcessor
nlp_processor = ChineseNLPProcessor()

def analyze_sentiment(text):
    """
    分析文本情感和关键词
    """
    # 清理文本
    cleaned = nlp_processor.text_cleanup(text)
    
    # 情感分析
    sentiment = nlp_processor.sentiment_analysis_simple(cleaned)
    
    # 提取关键词
    keywords = nlp_processor.extract_keywords(cleaned, top_n=5)
    keywords_str = ", ".join([f"{word}({weight:.3f})" for word, weight in keywords])
    
    # 分词
    words = nlp_processor.segment_text(cleaned)
    words_str = " / ".join(words)
    
    sentiment_map = {
        'positive': '正面 😊',
        'negative': '负面 😞',
        'neutral': '中性 😐'
    }
    
    return sentiment_map[sentiment], keywords_str, words_str

demo2 = gr.Interface(
    fn=analyze_sentiment,
    inputs=gr.Textbox(
        label="输入文本",
        placeholder="请输入要分析的中文文本...",
        lines=5
    ),
    outputs=[
        gr.Textbox(label="情感分析结果"),
        gr.Textbox(label="关键词"),
        gr.Textbox(label="分词结果")
    ],
    title="中文文本分析",
    description="分析中文文本的情感、关键词和分词",
    examples=[
        ["今天天气真不错，心情很好！"],
        ["这个产品质量太差了，非常失望。"],
        ["这是一个关于机器学习的介绍。"]
    ]
)

# demo2.launch()

### 示例3：文本相似度比较

In [ ]:
def compare_texts(text1, text2):
    """
    比较两段文本的相似度
    """
    similarity = nlp_processor.calculate_similarity(text1, text2)
    
    if similarity > 0.7:
        result = f"相似度: {similarity:.4f} - 非常相似 ✅"
    elif similarity > 0.4:
        result = f"相似度: {similarity:.4f} - 有一定相似 ⚠️"
    else:
        result = f"相似度: {similarity:.4f} - 不太相似 ❌"
    
    return result

demo3 = gr.Interface(
    fn=compare_texts,
    inputs=[
        gr.Textbox(label="文本1", placeholder="输入第一段文本", lines=3),
        gr.Textbox(label="文本2", placeholder="输入第二段文本", lines=3)
    ],
    outputs=gr.Textbox(label="相似度结果"),
    title="文本相似度分析",
    description="比较两段中文文本的相似程度",
    examples=[
        ["今天天气很好", "今天阳光明媚"],
        ["我喜欢吃苹果", "我喜欢吃香蕉"],
        ["机器学习很有趣", "今天下雨了"]
    ]
)

# demo3.launch()

### 组合多个界面

In [ ]:
# 使用Tab组合多个界面
combined_demo = gr.TabbedInterface(
    [demo1, demo2, demo3],
    ["问候应用", "情感分析", "相似度分析"],
    title="NLP工具集合"
)

# 启动组合界面
# combined_demo.launch(share=True)  # share=True可以生成公开链接

### Gradio关键特性

1. **简单易用**
   - 只需几行代码即可创建界面
   - 支持多种输入输出类型

2. **组件丰富**
   - Textbox, Slider, Dropdown, Radio等
   - 支持图像、音频、视频等多媒体

3. **分享功能**
   - `share=True`生成临时公开链接
   - 方便演示和分享模型

4. **示例功能**
   - 提供预设示例方便用户测试
   - 改善用户体验

## 总结

本notebook涵盖了NLP和大语言模型的多个重要主题：

1. **Python基础** - 理解函数参数的灵活使用
2. **中文NLP** - 掌握文本处理、分词、关键词提取等技术
3. **BERT微调** - 学习如何微调预训练模型用于特定任务
4. **LLaMA微调** - 了解使用LoRA高效微调大语言模型
5. **Gradio界面** - 快速为模型创建交互式Web界面

这些技术是现代NLP和AI应用开发的核心工具！